# Changing the simplex transform in Stan

Stan's built-in `simplex[K]` type always uses the same transform (a softmax of an isometric
log-ratio basis), and it cannot be swapped. To use a different transform:

1. declare an unconstrained parameter `vector[K - 1] theta_raw`,
2. build the simplex from it in `transformed parameters`,
3. add the transform's log-Jacobian with `jacobian +=` (Stan 2.36+).

ADVI then places its Normal on `theta_raw`. The two model files below have the same data, prior and
likelihood; only the parameterization of `theta` differs.

In [1]:
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modulars").is_dir())
STAN_DIR = REPO_ROOT / "single_MC" / "multinomial_dirichlet"

Default transform (`simplex[K] theta`):

In [2]:
print((STAN_DIR / "stan_dirichlet_multinomial.stan").read_text())

data {
  int<lower=1> N;
  int<lower=2> K;
  array[N] int<lower=0> total_count;
  array[N, K] int<lower=0> y;
  vector<lower=0>[K] alpha_prior;
}

parameters {
  simplex[K] theta;
}

model {
  theta ~ dirichlet(alpha_prior);
  for (n in 1:N) {
    y[n] ~ multinomial(theta);
  }
}



Changed transform (stick-breaking from `theta_raw`):

In [3]:
print((STAN_DIR / "stan_dirichlet_multinomial_stickbreaking.stan").read_text())

// Dirichlet-multinomial model with a user-defined simplex transform.
//
// Stan's built-in simplex[K] parameter uses softmax of an isometric log-ratio (ILR) basis.
// Here theta is instead built from unconstrained coordinates theta_raw with the centered
// stick-breaking transform used by NumPyro (StickBreakingTransform), TFP
// (IteratedSigmoidCentered) and our PyMC runs (StickBreakingSimplexTransform), so ADVI's
// mean-field Normal over theta_raw gives the same variational family as those packages.
// Data and likelihood are identical to stan_dirichlet_multinomial.stan.
data {
  int<lower=1> N;
  int<lower=2> K;
  array[N] int<lower=0> total_count;
  array[N, K] int<lower=0> y;
  vector<lower=0>[K] alpha_prior;
}
parameters {
  vector[K - 1] theta_raw;  // unconstrained stick-breaking coordinates
}
transformed parameters {
  simplex[K] theta;
  {
    real remaining = 1;
    for (k in 1:(K - 1)) {
      real u = theta_raw[k] - log(K - k);  // centering: theta_raw = 0 maps to the uni